# 03 · Ausblick: Was alles möglich ist

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LTplus-AG/bxmid03-hs26/blob/main/notebooks/2026-10-27_Automatisierung-I/03_Ausblick_Was_ist_moeglich.ipynb)

**BXMid03 · HS26 · Automatisierung I**

Dieses Notebook zeigt in kurzen, eigenständigen Beispielen, wie weit man mit ein paar Zeilen Python kommt:

1. **Prüfen und einfärben** – ein eigener 3D-Viewer, der jedes Bauteil nach Prüfergebnis färbt
2. **Grundriss aus der Geometrie** – ein Plan direkt aus dem IFC, ebenfalls mit Prüffarben
3. **Zurückschreiben** – das Prüfergebnis als Eigenschaft ins IFC speichern und weitergeben
4. **Zwei Modellstände vergleichen** – was hat sich zwischen zwei Lieferungen geändert?
5. **Bericht auf Knopfdruck** – eine HTML-Seite mit Kennzahlen, 3D und Tabelle

Die Geometrie liefert **IFC-Lite** (`ifclite-geom`): Dreiecksnetze in Metern, direkt in Python.

In [ ]:
#@title Installieren
!pip install -q ifcopenshell ifclite-geom plotly
print("Fertig.")

## Schritt 1 · Modell wählen

Diese Zelle ist ein **Formular**: rechts erscheint eine Auswahlliste statt Code.
Wenn du eine andere Auswahl triffst, läuft die Zelle automatisch neu. Danach die folgenden Zellen erneut ausführen
(oder *Laufzeit → Alle ausführen*).

In [ ]:
#@title Modell wählen { run: "auto" }
#@markdown Wähle ein Beispielmodell aus der Liste. Die Zelle lädt es aus dem Kurs-Repository auf GitHub.
#@markdown „Eigene Datei hochladen“ öffnet einen Dateidialog – nur Modelle verwenden, die das Projekt verlassen dürfen.
modell = "Tragwerk · BIMcollab-Beispiel (IFC2X3)" #@param ["Tragwerk · BIMcollab-Beispiel (IFC2X3)", "Brücke · IfcBridge-Beispiel (IFC4X3)", "Hochhaus · Snowdon Towers Tragwerk (IFC2X3)", "Holzrahmenwand · fehlerhaft (IFC4)", "Holzrahmenwand · korrigiert (IFC4)", "Eigene Datei hochladen"]

import urllib.request

BASIS = "https://raw.githubusercontent.com/LTplus-AG/bxmid03-hs26/main/modelle/"
DATEIEN = {
    "Tragwerk · BIMcollab-Beispiel (IFC2X3)": "Team_5_BIMcollab_Example_STR.ifc",
    "Brücke · IfcBridge-Beispiel (IFC4X3)": "Team_1_Bruecke_Arbeitsmodell.ifc",
    "Hochhaus · Snowdon Towers Tragwerk (IFC2X3)": "Team_4_Snowdon_Structural_Baufortschritt.ifc",
    "Holzrahmenwand · fehlerhaft (IFC4)": "Demo_Holzrahmen_fehlerhaft.ifc",
    "Holzrahmenwand · korrigiert (IFC4)": "Demo_Holzrahmen_korrigiert.ifc",
}

if modell == "Eigene Datei hochladen":
    from google.colab import files
    pfad = list(files.upload())[0]          # Dateidialog; die Datei liegt danach auf dem Google-Server
else:
    pfad = DATEIEN[modell]
    urllib.request.urlretrieve(BASIS + pfad, pfad)   # Datei aus GitHub herunterladen

print("Modell bereit:", pfad)

## 1 · Prüfen und in 3D einfärben

Eine Regel für Wände mit einstellbarem Grenzwert. Danach holt IFC-Lite die Geometrie aller Bauteile
und Plotly zeichnet sie: **grün** bestanden, **rot** durchgefallen, **orange** fehlend, **grau** nicht geprüft.

Schiebe den Regler und führe die Zelle neu aus: die Farben ändern sich.

In [ ]:
#@title Regel: Wandbreite mindestens …
mindestbreite_m = 0.15 #@param {type:"slider", min:0.0, max:0.5, step:0.01}

import ifcopenshell
import ifcopenshell.util.element as ue
import ifcopenshell.util.unit as uu
import pandas as pd

m = ifcopenshell.open(pfad)
faktor = uu.calculate_unit_scale(m)

def menge(element, name):
    """Sucht eine Menge (z. B. 'Width') in allen Mengen-Sets eines Bauteils.

    Mengen stehen je nach Software in 'Qto_WallBaseQuantities', 'BaseQuantities' o. ä.
    Wir suchen deshalb in allen Sets und nehmen den ersten Treffer.
    """
    for qset in ue.get_psets(element, qtos_only=True).values():
        if name in qset:
            return qset[name]
    return None

status = {}                                   # GlobalId -> Ergebnis
for w in m.by_type("IfcWall"):
    roh = menge(w, "Width")
    if roh is None:
        status[w.GlobalId] = "fehlend"
    else:
        status[w.GlobalId] = "bestanden" if roh * faktor >= mindestbreite_m else "durchgefallen"

print(f"{len(status)} Wände geprüft:", pd.Series(status).value_counts().to_dict() if status else "keine Wände im Modell")

In [ ]:
#@title Eigener 3D-Viewer mit Prüffarben (IFC-Lite-Geometrie + Plotly)
import numpy as np
import ifclite_geom
import plotly.graph_objects as go

geo = ifclite_geom.geometry_data_buffers(open(pfad, "rb").read())   # alle Bauteile als Dreiecksnetze, in Metern

FARBEN = {"bestanden": "#2e9e4f", "durchgefallen": "#d62728", "fehlend": "#ff9f1c", "nicht geprüft": "#c9ccd1"}

# Pro Farbe ein Netz: alle Dreiecke dieser Gruppe zusammenfügen (schneller als ein Netz pro Bauteil)
gruppen = {k: {"xyz": [], "ijk": [], "n": 0, "namen": []} for k in FARBEN}
for el in geo["elements"].values():
    if el["ifc_type"] in ("IfcOpeningElement", "IfcSpace"):
        continue
    gruppe = status.get(el["global_id"], "nicht geprüft")
    v = np.frombuffer(el["vertices"], dtype=np.float64).reshape(-1, 3)
    f = np.frombuffer(el["faces"], dtype=np.uint32).reshape(-1, 3)
    g = gruppen[gruppe]
    g["ijk"].append(f + g["n"]); g["xyz"].append(v); g["n"] += len(v)
    g["namen"] += [f'{el["ifc_type"]} · {el["name"]}'] * len(v)

fig = go.Figure()
for name, g in gruppen.items():
    if not g["xyz"]:
        continue
    xyz, ijk = np.vstack(g["xyz"]), np.vstack(g["ijk"])
    fig.add_mesh3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], i=ijk[:, 0], j=ijk[:, 1], k=ijk[:, 2],
                   color=FARBEN[name], opacity=0.35 if name == "nicht geprüft" else 1.0,
                   name=name, showlegend=True, hovertext=g["namen"], hoverinfo="text+name", flatshading=True)
fig.update_layout(scene=dict(aspectmode="data"), height=650, margin=dict(l=0, r=0, t=30, b=0),
                  title=f"Wandbreite ≥ {mindestbreite_m} m")
fig.show()

## 2 · Grundriss aus der Geometrie

Ein Grundriss ist eine Projektion von oben. Wir nehmen alle Dreiecke eines Höhenbereichs,
legen sie flach (z weglassen) und zeichnen sie mit **matplotlib** – Wände in Prüffarben.

In [ ]:
#@title Grundriss eines Höhenbereichs
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection

# Höhenbereich aus der Geometrie vorschlagen: unterstes Drittel des Modells
alle_z = np.concatenate([np.frombuffer(e["vertices"], dtype=np.float64)[2::3] for e in geo["elements"].values()])
z_min, z_max = float(alle_z.min()), float(alle_z.max())
von_m = z_min + 0.1 * (z_max - z_min)   # Schnitthöhe von …
bis_m = z_min + 0.4 * (z_max - z_min)   # … bis

hintergrund, waende, farben = [], [], []        # zwei Ebenen: andere Bauteile hell, Wände farbig darüber
for el in geo["elements"].values():
    if el["ifc_type"] in ("IfcOpeningElement", "IfcSpace"):
        continue
    v = np.frombuffer(el["vertices"], dtype=np.float64).reshape(-1, 3)
    if v[:, 2].max() < von_m or v[:, 2].min() > bis_m:      # liegt nicht im Höhenbereich
        continue
    f = np.frombuffer(el["faces"], dtype=np.uint32).reshape(-1, 3)
    dreiecke = list(v[f][:, :, :2])                        # Dreiecke ohne z = Projektion von oben
    if el["global_id"] in status:
        waende += dreiecke
        farben += [FARBEN[status[el["global_id"]]]] * len(dreiecke)
    else:
        hintergrund += dreiecke

fig2, ax = plt.subplots(figsize=(11, 8))
ax.add_collection(PolyCollection(hintergrund, facecolors="#e6e8eb", edgecolors="none"))
ax.add_collection(PolyCollection(waende, facecolors=farben, edgecolors="none"))
ax.autoscale(); ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
ax.set_title(f"Grundriss {von_m:.1f} – {bis_m:.1f} m · Wände nach Prüfergebnis")
plt.show()

## 3 · Ergebnis ins Modell zurückschreiben

Statt einer separaten Liste schreiben wir das Ergebnis **als Property Set** an jede Wand.
Die neue Datei lässt sich in jedem IFC-Viewer öffnen – z. B. in [ifclite.com](https://www.ifclite.com)
nach `BXMid03_Pruefung.Status` einfärben oder filtern.

`ifcopenshell.api` ist die „saubere“ Art, ein Modell zu verändern (wie ein Klick in einer BIM-Software).

In [ ]:
#@title Pset „BXMid03_Pruefung“ an alle geprüften Wände schreiben
import datetime
import ifcopenshell.api

heute = datetime.date.today().isoformat()
for gid, ergebnis in status.items():
    wand = m.by_guid(gid)
    pset = ifcopenshell.api.run("pset.add_pset", m, product=wand, name="BXMid03_Pruefung")
    ifcopenshell.api.run("pset.edit_pset", m, pset=pset, properties={
        "Status": ergebnis,
        "Regel": f"Width >= {mindestbreite_m} m",
        "Datum": heute,
    })

neu = pfad.replace(".ifc", "_geprueft.ifc")
m.write(neu)

# Gegenprobe: neue Datei öffnen und eine Wand lesen
kontrolle = ifcopenshell.open(neu)
beispiel = next(iter(status), None)
if beispiel:
    print("Gespeichert:", neu)
    print("Beispiel:", ue.get_psets(kontrolle.by_guid(beispiel))["BXMid03_Pruefung"])

In [ ]:
#@title Geprüftes IFC herunterladen
from google.colab import files
files.download(neu)

## 4 · Zwei Modellstände vergleichen

Typische Frage nach einer neuen Lieferung: *Was hat sich geändert?*
Wir laden zwei Stände desselben Modells (Holzrahmenwand fehlerhaft → korrigiert),
ordnen die Bauteile über die **GlobalId** einander zu und vergleichen alle Eigenschaften und Mengen.

In [ ]:
#@title Stand A und Stand B vergleichen
import urllib.request

alt_datei, neu_datei = "Demo_Holzrahmen_fehlerhaft.ifc", "Demo_Holzrahmen_korrigiert.ifc"
for d in (alt_datei, neu_datei):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/LTplus-AG/bxmid03-hs26/main/modelle/" + d, d)

def eigenschaften(datei):
    """{GlobalId: {'Set.Eigenschaft': Wert}} für alle Bauteile einer Datei."""
    mod = ifcopenshell.open(datei)
    aus = {}
    for b in mod.by_type("IfcElement"):
        flach = {}
        for s, werte in ue.get_psets(b).items():
            for k, v in werte.items():
                if k != "id":
                    flach[f"{s}.{k}"] = v
        aus[b.GlobalId] = flach
    return aus

A, B = eigenschaften(alt_datei), eigenschaften(neu_datei)
aenderungen = []
for gid in sorted(set(A) | set(B)):
    if gid not in A: aenderungen.append({"GlobalId": gid, "Was": "neu", "Eigenschaft": "", "Vorher": "", "Nachher": ""}); continue
    if gid not in B: aenderungen.append({"GlobalId": gid, "Was": "gelöscht", "Eigenschaft": "", "Vorher": "", "Nachher": ""}); continue
    for k in sorted(set(A[gid]) | set(B[gid])):
        if A[gid].get(k) != B[gid].get(k):
            aenderungen.append({"GlobalId": gid, "Was": "geändert", "Eigenschaft": k,
                                "Vorher": A[gid].get(k), "Nachher": B[gid].get(k)})

print(f"{len(A)} Bauteile vorher, {len(B)} nachher, {len(aenderungen)} Änderungen")
pd.DataFrame(aenderungen)

## 5 · Bericht auf Knopfdruck

Alles zusammen in **eine HTML-Datei**: Kennzahlen, das 3D-Modell (drehbar, auch offline im Browser)
und die Tabelle der Problemfälle. Diese Datei kann man per Mail verschicken oder ins Repository legen.

In [ ]:
#@title HTML-Bericht erzeugen und herunterladen
from google.colab import files
zaehlung = pd.Series(status).value_counts().to_dict() if status else {}
probleme = pd.DataFrame(
    [{"GlobalId": g, "Name": m.by_guid(g).Name, "Ergebnis": e} for g, e in status.items() if e != "bestanden"])

html = f"""<!doctype html><html lang="de"><meta charset="utf-8">
<title>Prüfbericht {pfad}</title>
<body style="font-family:sans-serif;max-width:1100px;margin:auto">
<h1>Prüfbericht</h1>
<p><b>Modell:</b> {pfad} · <b>Schema:</b> {m.schema} · <b>Regel:</b> Wandbreite ≥ {mindestbreite_m} m · <b>Datum:</b> {heute}</p>
<h2>Ergebnis</h2><p>{" · ".join(f"{k}: <b>{v}</b>" for k, v in zaehlung.items()) or "keine Wände"}</p>
<h2>3D</h2>{fig.to_html(full_html=False, include_plotlyjs="cdn")}
<h2>Problemfälle</h2>{probleme.to_html(index=False) if len(probleme) else "<p>Keine.</p>"}
</body></html>"""

open("pruefbericht.html", "w", encoding="utf-8").write(html)
files.download("pruefbericht.html")
print("Bericht erstellt: pruefbericht.html")

## Und noch weiter …

Ideen, die mit denselben Bausteinen möglich sind:

| Idee | Bausteine |
|---|---|
| **BCF-Themen** automatisch aus Prüffehlern erzeugen | `ifcopenshell.bcf` |
| **Mengen und Kosten** je Geschoss oder Bauabschnitt | Tabelle aus Notebook 02 + Preisliste (CSV) |
| **Termine (4D)**: Bauteile nach Datum einfärben, Animation | Psets mit Datum + Plotly `animation_frame` |
| **Ökobilanz**: Volumen × CO₂-Faktor je Material | Tabelle + KBOB-Liste |
| **Regeln aus Excel** lesen und automatisch als IDS schreiben | `pandas.read_excel` + `ifctester.ids` |
| **Gemini in Colab** fragen: „Schreib mir eine Zelle, die …“ | ✦-Symbol in Colab – Code immer prüfen! |

> Was ihr davon baut, gehört ins **Team-Repository**: Notebook speichern mit
> *Datei → Kopie in GitHub speichern*.